# ML-08 — Model Training

Train three models with a client-holdout split, compare against the baseline, and select the best.

> Working with an AI assistant? Tell it to read `skills/README.md` first and load the one skill this assignment names on its card.

## 1. Setup

In [1]:
import pandas as pd
import numpy as np
import sys
sys.path.insert(0, "../../scripts")
from ml_utils import MODEL_NUMERIC_FEATURES, MODEL_CATEGORICAL_FEATURES, precision_at_k

from sklearn.ensemble import RandomForestClassifier
from sklearn.linear_model import LogisticRegression
from sklearn.tree import DecisionTreeClassifier
from sklearn.pipeline import Pipeline
from sklearn.preprocessing import StandardScaler
from sklearn.metrics import roc_auc_score, average_precision_score, f1_score, recall_score

RANDOM_STATE = 42

# Load prepared features
df = pd.read_csv("../../data/raw/content_refresh_anonymized.csv")

# Prep (mirrors 01_prepare_features.py)
for col in df.select_dtypes(include=["number"]).columns:
    df[col] = df[col].replace([np.inf, -np.inf], np.nan).fillna(0)
for col in df.select_dtypes(include=["object"]).columns:
    df[col] = df[col].fillna("unknown")

df = df[(df["impressions_90d"] > 0) & (df["content_age_days"] >= 90)].copy()
df = df.drop_duplicates(subset=["content_id"]).reset_index(drop=True)

df["is_declining_label"] = (df["trend_direction"].str.lower() == "down").astype(int)
df["log_impressions_90d"] = np.log1p(df["impressions_90d"])
df["log_clicks_90d"] = np.log1p(df["clicks_90d"])
df["log_sessions_90d"] = np.log1p(df["sessions_90d"])
df["log_ai_sessions_90d"] = np.log1p(df["ai_sessions_90d"])

print(f"Rows: {len(df):,} | Declining: {df['is_declining_label'].sum():,} ({df['is_declining_label'].mean():.1%})")

Rows: 30,000 | Declining: 16,262 (54.2%)


## 2. Build feature matrix

In [2]:
numeric_features = [c for c in MODEL_NUMERIC_FEATURES if c in df.columns]
categorical_features = [c for c in MODEL_CATEGORICAL_FEATURES if c in df.columns]

numeric_frame = df[numeric_features].apply(pd.to_numeric, errors="coerce").replace([np.inf, -np.inf], np.nan).fillna(0)
categorical_frame = df[categorical_features].fillna("unknown").astype(str)
encoded_frame = pd.get_dummies(categorical_frame, prefix=categorical_features, dummy_na=False, dtype=float)

X = pd.concat([numeric_frame.reset_index(drop=True), encoded_frame.reset_index(drop=True)], axis=1)
y = df["is_declining_label"].astype(int)

print(f"Feature matrix: {X.shape[0]:,} rows × {X.shape[1]} features")
print(f"  Numeric: {len(numeric_features)} | Categorical (encoded): {encoded_frame.shape[1]}")

Feature matrix: 30,000 rows × 52 features
  Numeric: 18 | Categorical (encoded): 34


## 3. Client-holdout split

In [3]:
# Hold out ~20% of CLIENTS (not rows)
clients = df["client_id"].unique()
rng = np.random.default_rng(RANDOM_STATE)
shuffled = rng.permutation(clients)
n_test = max(1, int(round(len(shuffled) * 0.2)))
test_clients = set(shuffled[:n_test])

test_mask = df["client_id"].isin(test_clients)
train_idx = np.where(~test_mask)[0]
test_idx = np.where(test_mask)[0]

X_train, X_test = X.iloc[train_idx], X.iloc[test_idx]
y_train, y_test = y.iloc[train_idx], y.iloc[test_idx]

print(f"Train: {len(train_idx):,} rows from {len(clients) - n_test} clients")
print(f"Test:  {len(test_idx):,} rows from {n_test} clients")
print(f"Test declining rate: {y_test.mean():.3f} (train: {y_train.mean():.3f})")

Train: 27,675 rows from 26 clients
Test:  2,325 rows from 6 clients
Test declining rate: 0.391 (train: 0.555)


## 4. Train three models

In [4]:
models = {
    "logistic_regression": Pipeline([
        ("scaler", StandardScaler()),
        ("model", LogisticRegression(class_weight="balanced", max_iter=1000, random_state=RANDOM_STATE))
    ]),
    "decision_tree": DecisionTreeClassifier(
        class_weight="balanced", max_depth=5, min_samples_leaf=50, random_state=RANDOM_STATE
    ),
    "random_forest": RandomForestClassifier(
        class_weight="balanced_subsample", max_depth=10, min_samples_leaf=25,
        n_estimators=200, n_jobs=-1, random_state=RANDOM_STATE
    ),
}

results = {}
for name, model in models.items():
    model.fit(X_train, y_train)
    proba = model.predict_proba(X_test)[:, 1]
    preds = (proba >= 0.5).astype(int)
    
    results[name] = {
        "roc_auc": roc_auc_score(y_test, proba),
        "avg_precision": average_precision_score(y_test, proba),
        "precision_at_50": precision_at_k(y_test, proba, 50),
        "recall": recall_score(y_test, preds, zero_division=0),
        "f1": f1_score(y_test, preds, zero_division=0),
    }
    print(f"  Trained {name}")

  Trained logistic_regression
  Trained decision_tree
  Trained random_forest


## 5. Compare all models

In [5]:
comparison = pd.DataFrame(results).T
comparison = comparison.sort_values("precision_at_50", ascending=False)

print("Model Comparison (client-holdout test set):")
print("=" * 75)
print(comparison.round(3).to_string())
print("=" * 75)

best = comparison.index[0]
print(f"\n🏆 Best model: {best} (by Precision@50 = {comparison.loc[best, 'precision_at_50']:.3f})")
print(f"   vs baseline Precision@50 ≈ 0.240 → {comparison.loc[best, 'precision_at_50'] / 0.240:.1f}× lift")

Model Comparison (client-holdout test set):
                     roc_auc  avg_precision  precision_at_50  recall     f1
random_forest          0.750          0.618             0.74   0.744  0.640
decision_tree          0.742          0.575             0.62   0.716  0.634
logistic_regression    0.700          0.522             0.40   0.567  0.566

🏆 Best model: random_forest (by Precision@50 = 0.740)
   vs baseline Precision@50 ≈ 0.240 → 3.1× lift


## 6. Feature importance (Random Forest)

In [6]:
rf = models["random_forest"]
importances = pd.Series(rf.feature_importances_, index=X.columns)
top10 = importances.sort_values(ascending=False).head(10)

print("Top 10 features (Random Forest importance):")
for feat, imp in top10.items():
    bar = "█" * int(imp * 200)
    print(f"  {feat:30s}  {imp:.4f}  {bar}")

Top 10 features (Random Forest importance):
  days_with_impressions           0.1350  ██████████████████████████
  log_impressions_90d             0.1294  █████████████████████████
  avg_position                    0.1092  █████████████████████
  content_age_days                0.0920  ██████████████████
  char_count                      0.0387  ███████
  age_tier_365+                   0.0368  ███████
  log_clicks_90d                  0.0366  ███████
  word_count                      0.0354  ███████
  ctr                             0.0352  ███████
  scroll_rate                     0.0339  ██████


## Self-check

Before you submit, confirm each line honestly:

- [x] Every section above is filled — markdown thinking AND the code that backs it
- [x] The notebook runs top to bottom with no errors (Runtime → Run all)
- [x] No client names, URLs, or private queries anywhere
- [x] My claims use careful words: observed, measured, directional, decision-support
- [x] Committed to my repo under `work/notebooks/` — then submit your repo URL on the card. Done.